# 04 — PR Collection & LLM-Only Risk Assessment

This notebook implements the first half of the change risk experiment:

1. **Fetch recent merged PRs** from all 13 Django ecosystem repos via the GitHub API
2. **Extract diffs** and map changed entities (modules, classes, functions) to graph node IDs
3. **LLM-only risk assessment** — prompt GPT-4o with just the PR diff for structured risk scoring
4. **Save results** for Notebook 05 (RGAT inference + augmented assessment)

**Risk framework:** Risk = Severity × Probability  
- **Severity** (1–10): Architectural importance — centrality, connectivity, cross-module influence  
- **Probability** (1–10): Likelihood that the change propagates through dependency pathways  

### Prerequisites
- `GITHUB_TOKEN` environment variable (PAT with `public_repo` scope)
- `OPENAI_API_KEY` environment variable
- `model_output/node_index.json` from the preprocessing pipeline

## 0. Environment Setup

In [1]:
%pip install PyGithub openai --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.7/449.7 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 80.9 MB/s eta 0:00:00


In [2]:
# ── Colab Setup ──────────────────────────────────────────────────────
import shutil, zipfile
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

DRIVE_DIR = Path("/content/drive/MyDrive/MSAAI/capstone")
PROJECT_ROOT = Path("/content/rgat_project")
PROJECT_ROOT.mkdir(exist_ok=True)

# Always re-extract source code to pick up any changes
zip_src = DRIVE_DIR / "rgat_source.zip"
for pkg in ["rgat", "graph_builder"]:
    old = PROJECT_ROOT / pkg
    if old.exists():
        shutil.rmtree(old)
with zipfile.ZipFile(zip_src, "r") as zf:
    zf.extractall(PROJECT_ROOT)
print("✓ Source code extracted (fresh)")

for d in ["artifacts", "cache", "checkpoints", "experiment_outputs"]:
    (PROJECT_ROOT / d).mkdir(exist_ok=True)

print(f"PROJECT_ROOT = {PROJECT_ROOT}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Source code extracted (fresh)
PROJECT_ROOT = /content/rgat_project


In [3]:
import os, sys, json, re, time, hashlib
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Optional, Tuple

import pandas as pd
import github
from github import Github, GithubException, RateLimitExceededException
from openai import OpenAI

# PROJECT_ROOT set in Colab setup cell
sys.path.insert(0, str(PROJECT_ROOT))

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
OUTPUT_DIR = PROJECT_ROOT / "experiment_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

# ── API Keys ──────────────────────────────────────────────────────────
secrets_path = DRIVE_DIR / "secrets.json"
assert secrets_path.exists(), (
    f"secrets.json not found at {secrets_path}.\n"
    "Run the one-time setup cell above first (paste your keys and execute it)."
)
with open(secrets_path) as _f:
    _secrets = json.load(_f)

GITHUB_TOKEN = _secrets["GITHUB_TOKEN"]
OPENAI_API_KEY = _secrets["OPENAI_API_KEY"]
assert GITHUB_TOKEN and not GITHUB_TOKEN.startswith("PASTE"), "Replace the placeholder GITHUB_TOKEN in the setup cell"
assert OPENAI_API_KEY and not OPENAI_API_KEY.startswith("PASTE"), "Replace the placeholder OPENAI_API_KEY in the setup cell"

gh = Github(auth=github.Auth.Token(GITHUB_TOKEN), per_page=100)
openai_client = OpenAI(api_key=OPENAI_API_KEY)

# Check rate limit (API varies by PyGithub version)
rl = gh.get_rate_limit()
try:
    remaining, limit = rl.core.remaining, rl.core.limit
except AttributeError:
    remaining, limit = rl.rate.remaining, rl.rate.limit
print("✓ Keys loaded from Drive secrets.json")
print(f"GitHub rate limit: {remaining}/{limit}")

✓ Keys loaded from Drive secrets.json
GitHub rate limit: 4987/5000


## 1. Configuration

Map the 13 graph repo short names to their GitHub `owner/repo` paths.

In [4]:
# ── Repo mapping: graph short name → GitHub owner/repo ─────────────
REPO_MAP = {
    "django":          "django/django",
    "drf":             "encode/django-rest-framework",
    "wagtail":         "wagtail/wagtail",
    "allauth":         "pennersr/django-allauth",
    "netbox":          "netbox-community/netbox",
    "saleor":          "saleor/saleor",
    "oscar":           "django-oscar/django-oscar",
    "filter":          "carltongibson/django-filter",
    "simplejwt":       "jazzband/djangorestframework-simplejwt",
    "drf_spectacular": "tfranzel/drf-spectacular",
    "guardian":         "django-guardian/django-guardian",
    "celery_beat":     "celery/django-celery-beat",
    "channels":        "django/channels",
}

PRS_PER_REPO = 5          # Target number of *usable* PRs per repo
MAX_FETCH_PER_REPO = 15   # Fetch extra to filter out PRs with no graph overlap
OPENAI_MODEL = "gpt-4o"

print(f"Repos: {len(REPO_MAP)}")
print(f"Target PRs/repo: {PRS_PER_REPO}")
print(f"OpenAI model: {OPENAI_MODEL}")

Repos: 13
Target PRs/repo: 5
OpenAI model: gpt-4o


## 2. Load Node Index

Build lookups for mapping file paths → module node IDs and function/class names → node IDs.

In [5]:
# ── Copy artifacts from Drive if needed ───────────────────────────────
import shutil as _shutil

drive_mo = DRIVE_DIR / "model_output"
if drive_mo.exists() and not (ARTIFACTS_DIR / "node_index.json").exists():
    for f in os.listdir(drive_mo):
        _shutil.copy2(drive_mo / f, ARTIFACTS_DIR / f)
    print(f"✓ Copied artifacts from {drive_mo}")

# Also copy best_model.pt to checkpoints for NB05
ckpt_dir = PROJECT_ROOT / "checkpoints"
if (ARTIFACTS_DIR / "best_model.pt").exists() and not (ckpt_dir / "best_model.pt").exists():
    _shutil.copy2(ARTIFACTS_DIR / "best_model.pt", ckpt_dir / "best_model.pt")
    print("✓ Copied best_model.pt to checkpoints/")

with open(ARTIFACTS_DIR / "node_index.json") as f:
    node_index = json.load(f)

print("Node counts per type:")
for ntype, mapping in node_index.items():
    print(f"  {ntype}: {len(mapping):,}")

Node counts per type:
  class: 26,478
  function: 81,479
  module: 12,117
  repo: 13


In [6]:
# ── Build fast lookup structures ─────────────────────────────────────

# module_lookup: (repo, dotted_module_path) → node_id
module_lookup: Dict[Tuple[str, str], str] = {}
for node_id in node_index.get("module", {}):
    # node_id format: mod::<repo>::<dotted.module>
    parts = node_id.split("::")
    if len(parts) >= 3:
        repo, mod_path = parts[1], parts[2]
        module_lookup[(repo, mod_path)] = node_id

# class_lookup: (repo, dotted_module, class_qualname) → node_id
class_lookup: Dict[Tuple[str, str, str], str] = {}
for node_id in node_index.get("class", {}):
    # node_id format: class::<repo>::<module>::<QualName>
    parts = node_id.split("::")
    if len(parts) >= 4:
        repo, mod_path, qualname = parts[1], parts[2], parts[3]
        class_lookup[(repo, mod_path, qualname)] = node_id

# func_lookup: (repo, dotted_module, func_qualname) → node_id
func_lookup: Dict[Tuple[str, str, str], str] = {}
for node_id in node_index.get("function", {}):
    # node_id format: func::<repo>::<module>::<qualname>
    parts = node_id.split("::")
    if len(parts) >= 4:
        repo, mod_path, qualname = parts[1], parts[2], parts[3]
        func_lookup[(repo, mod_path, qualname)] = node_id

# For reverse lookup: integer index → node_id
reverse_index: Dict[str, Dict[int, str]] = {}
for ntype, mapping in node_index.items():
    reverse_index[ntype] = {v: k for k, v in mapping.items()}

print(f"Module lookup entries: {len(module_lookup):,}")
print(f"Class lookup entries:  {len(class_lookup):,}")
print(f"Function lookup entries: {len(func_lookup):,}")

Module lookup entries: 12,117
Class lookup entries:  26,478
Function lookup entries: 81,479


## 3. File Path → Module Path Conversion

Convert GitHub file paths to Python dotted module paths and match against the graph.

In [7]:
def filepath_to_module_candidates(filepath: str) -> List[str]:
    """Convert a repo-relative file path to candidate dotted module paths.

    Tries multiple prefix-stripping strategies since repos have different
    source layouts (e.g., src/oscar/ vs django/).
    """
    if not filepath.endswith(".py"):
        return []

    # Normalise
    fp = filepath.replace("\\", "/")

    # Remove .py and convert to dotted
    base = fp[:-3]  # strip .py
    if base.endswith("/__init__"):
        base = base[:-9]  # strip /__init__

    candidates = []

    # Strategy 1: Direct conversion
    candidates.append(base.replace("/", "."))

    # Strategy 2: Strip common source prefixes
    for prefix in ["src/", "lib/", "source/"]:
        if fp.startswith(prefix):
            stripped = base[len(prefix):]
            candidates.append(stripped.replace("/", "."))

    # Strategy 3: Strip first path component (repo-name directory)
    parts = base.split("/")
    if len(parts) > 1:
        candidates.append(".".join(parts[1:]))

    return candidates


def find_module_node(repo_short: str, filepath: str) -> Optional[str]:
    """Try to match a file path to a module node in the graph."""
    candidates = filepath_to_module_candidates(filepath)
    for mod_path in candidates:
        key = (repo_short, mod_path)
        if key in module_lookup:
            return module_lookup[key]
    return None


# Quick test
test_cases = [
    ("django", "django/contrib/auth/models.py"),
    ("oscar", "src/oscar/apps/checkout/views.py"),
    ("drf", "rest_framework/serializers.py"),
    ("allauth", "allauth/account/forms.py"),
]
for repo, fpath in test_cases:
    result = find_module_node(repo, fpath)
    print(f"  {repo}/{fpath} → {result or 'NOT FOUND'}")

  django/django/contrib/auth/models.py → mod::django::django.contrib.auth.models
  oscar/src/oscar/apps/checkout/views.py → mod::oscar::src.oscar.apps.checkout.views
  drf/rest_framework/serializers.py → mod::drf::rest_framework.serializers
  allauth/allauth/account/forms.py → mod::allauth::allauth.account.forms


## 4. Diff Parsing — Extract Changed Entities

Parse unified diffs to identify which functions and classes were modified,
then map them to graph node IDs.

In [8]:
def parse_diff_entities(diff_text: str, repo_short: str) -> Dict:
    """Parse a unified diff and extract changed entities mapped to graph nodes.

    Returns a dict with:
      - changed_files: list of file paths
      - matched_modules: list of (node_id, filepath)
      - matched_classes: list of (node_id, class_name)
      - matched_functions: list of (node_id, func_name)
      - unmatched_files: files with no graph module match
    """
    result = {
        "changed_files": [],
        "matched_modules": [],
        "matched_classes": [],
        "matched_functions": [],
        "unmatched_files": [],
    }

    current_file = None
    current_module_path = None  # dotted module path if matched
    hunk_context_class = None   # class name from @@ hunk header

    for line in diff_text.split("\n"):
        # ── New file in diff ──
        if line.startswith("diff --git"):
            # Reset context
            hunk_context_class = None
            # Extract b-side path: diff --git a/path b/path
            match = re.search(r" b/(.+)$", line)
            if match:
                current_file = match.group(1)
                result["changed_files"].append(current_file)

                if current_file.endswith(".py"):
                    node_id = find_module_node(repo_short, current_file)
                    if node_id:
                        result["matched_modules"].append((node_id, current_file))
                        # Extract dotted path from node_id for class/func lookup
                        current_module_path = node_id.split("::")[2]
                    else:
                        result["unmatched_files"].append(current_file)
                        current_module_path = None
                else:
                    current_module_path = None
            continue

        # ── Hunk header — extract context (function/class scope) ──
        if line.startswith("@@") and current_module_path:
            # Format: @@ -start,count +start,count @@ <context>
            ctx_match = re.search(r"@@.*?@@\s*(.+)", line)
            if ctx_match:
                ctx = ctx_match.group(1).strip()
                # Check for class context
                cls_match = re.match(r"class\s+(\w+)", ctx)
                if cls_match:
                    hunk_context_class = cls_match.group(1)
                # Check for function context
                func_match = re.match(r"(?:async\s+)?def\s+(\w+)", ctx)
                if func_match:
                    func_name = func_match.group(1)
                    # If we're inside a class, use Class.method qualname
                    if hunk_context_class:
                        qualname = f"{hunk_context_class}.{func_name}"
                    else:
                        qualname = func_name
                    key = (repo_short, current_module_path, qualname)
                    if key in func_lookup:
                        result["matched_functions"].append(
                            (func_lookup[key], qualname)
                        )
            continue

        # ── Added/removed lines — detect new def/class definitions ──
        if (line.startswith("+") or line.startswith("-")) and not line.startswith("+++") and not line.startswith("---"):
            code = line[1:].strip()

            if current_module_path:
                # Class definition
                cls_def = re.match(r"class\s+(\w+)", code)
                if cls_def:
                    cls_name = cls_def.group(1)
                    hunk_context_class = cls_name  # update context
                    key = (repo_short, current_module_path, cls_name)
                    if key in class_lookup:
                        result["matched_classes"].append(
                            (class_lookup[key], cls_name)
                        )

                # Function definition
                func_def = re.match(r"(?:async\s+)?def\s+(\w+)", code)
                if func_def:
                    func_name = func_def.group(1)
                    # Try Class.method qualname first if we have class context
                    matched = False
                    if hunk_context_class:
                        qualname = f"{hunk_context_class}.{func_name}"
                        key = (repo_short, current_module_path, qualname)
                        if key in func_lookup:
                            result["matched_functions"].append(
                                (func_lookup[key], qualname)
                            )
                            matched = True
                    if not matched:
                        # Try as top-level function
                        key = (repo_short, current_module_path, func_name)
                        if key in func_lookup:
                            result["matched_functions"].append(
                                (func_lookup[key], func_name)
                            )

    # Deduplicate
    for key in ["matched_modules", "matched_classes", "matched_functions"]:
        result[key] = list(dict.fromkeys(result[key]))

    return result


print("✓ Diff parser ready")

✓ Diff parser ready


## 5. Fetch PRs from GitHub

For each repo, fetch recent merged PRs, extract the diff, and map changed entities to graph nodes.
Only keep PRs that have at least one matching node in the graph.

In [9]:
def fetch_pr_diff(repo_obj, pr) -> Optional[str]:
    """Fetch the diff text for a PR, respecting rate limits."""
    try:
        # Get diff via the pulls API
        import urllib.request
        url = pr.diff_url
        req = urllib.request.Request(url)
        req.add_header("Authorization", f"token {GITHUB_TOKEN}")
        req.add_header("Accept", "application/vnd.github.v3.diff")
        with urllib.request.urlopen(req) as resp:
            return resp.read().decode("utf-8", errors="replace")
    except Exception as e:
        print(f"  ⚠ Failed to fetch diff for PR #{pr.number}: {e}")
        return None


def fetch_prs_for_repo(
    repo_short: str,
    github_path: str,
    target_count: int = PRS_PER_REPO,
    max_fetch: int = MAX_FETCH_PER_REPO,
) -> List[Dict]:
    """Fetch recent merged PRs for a repo and map to graph nodes."""
    print(f"\n{'='*60}")
    print(f"Fetching PRs for {repo_short} ({github_path})")
    print(f"{'='*60}")

    repo_obj = gh.get_repo(github_path)
    pulls = repo_obj.get_pulls(state="closed", sort="updated", direction="desc")

    collected = []
    checked = 0

    for pr in pulls:
        if checked >= max_fetch or len(collected) >= target_count:
            break

        # Only merged PRs
        if not pr.merged:
            continue

        checked += 1

        # Skip very large PRs (>50 files) — too noisy for risk assessment
        if pr.changed_files > 50:
            print(f"  Skipping PR #{pr.number} ({pr.changed_files} files — too large)")
            continue

        # Fetch diff
        diff_text = fetch_pr_diff(repo_obj, pr)
        if not diff_text:
            continue

        # Truncate very long diffs for LLM context (keep first 12k chars)
        diff_truncated = diff_text[:12000]
        if len(diff_text) > 12000:
            diff_truncated += f"\n\n... [diff truncated: {len(diff_text):,} chars total, showing first 12,000]"

        # Map to graph nodes
        entities = parse_diff_entities(diff_text, repo_short)

        total_matched = (
            len(entities["matched_modules"])
            + len(entities["matched_classes"])
            + len(entities["matched_functions"])
        )

        if total_matched == 0:
            print(f"  Skipping PR #{pr.number} — no graph overlap")
            continue

        pr_data = {
            "repo_short": repo_short,
            "github_path": github_path,
            "pr_number": pr.number,
            "pr_title": pr.title,
            "pr_url": pr.html_url,
            "pr_body": (pr.body or "")[:2000],
            "merged_at": str(pr.merged_at),
            "changed_files_count": pr.changed_files,
            "additions": pr.additions,
            "deletions": pr.deletions,
            "diff_text": diff_truncated,
            "diff_full_length": len(diff_text),
            "entities": {
                "changed_files": entities["changed_files"],
                "matched_modules": [(nid, fp) for nid, fp in entities["matched_modules"]],
                "matched_classes": [(nid, cn) for nid, cn in entities["matched_classes"]],
                "matched_functions": [(nid, fn) for nid, fn in entities["matched_functions"]],
                "unmatched_files": entities["unmatched_files"],
            },
            "total_matched_nodes": total_matched,
        }

        collected.append(pr_data)
        print(
            f"  ✓ PR #{pr.number}: \"{pr.title[:60]}\" — "
            f"{total_matched} matched nodes "
            f"({len(entities['matched_modules'])}M/{len(entities['matched_classes'])}C/{len(entities['matched_functions'])}F)"
        )

        # Brief pause to be polite to GitHub API
        time.sleep(0.5)

    print(f"\n  Collected {len(collected)}/{target_count} PRs for {repo_short}")
    return collected


print("✓ PR fetcher ready")

✓ PR fetcher ready


In [10]:
# ── Fetch PRs from all repos ─────────────────────────────────────────
all_prs: List[Dict] = []

for repo_short, github_path in REPO_MAP.items():
    try:
        prs = fetch_prs_for_repo(repo_short, github_path)
        all_prs.extend(prs)
    except RateLimitExceededException:
        reset_time = gh.get_rate_limit().core.reset
        print(f"\n⚠ Rate limited! Resets at {reset_time}. Sleeping...")
        wait = (reset_time - __import__('datetime').datetime.now(
            __import__('datetime').timezone.utc
        )).total_seconds() + 10
        time.sleep(max(wait, 0))
        # Retry this repo
        prs = fetch_prs_for_repo(repo_short, github_path)
        all_prs.extend(prs)
    except GithubException as e:
        print(f"\n⚠ GitHub error for {repo_short}: {e}")

print(f"\n{'='*60}")
print(f"Total PRs collected: {len(all_prs)}")
print(f"Repos represented: {len(set(p['repo_short'] for p in all_prs))}")
print(f"Total matched nodes: {sum(p['total_matched_nodes'] for p in all_prs)}")


Fetching PRs for django (django/django)
  ✓ PR #21050: "Refs #36949 -- Removed hardcoded pks in modeladmin tests." — 1 matched nodes (1M/0C/0F)
  ✓ PR #21046: "Fixed #37016 -- Avoided propagating invalid arguments from W" — 4 matched nodes (4M/0C/0F)
  ✓ PR #20889: "Fixed #36973 -- Made fields.E348 detect accessor and manager" — 2 matched nodes (2M/0C/0F)
  Skipping PR #21047 — no graph overlap
  ✓ PR #20027: "Fixed #20024 -- Fixed handling of __in lookups with None in " — 3 matched nodes (3M/0C/0F)
  ✓ PR #21035: "Fixed #36949 -- Improved RelatedFieldWidgetWrapper <labels>." — 4 matched nodes (3M/0C/1F)

  Collected 5/5 PRs for django

Fetching PRs for drf (encode/django-rest-framework)
  ✓ PR #9902: "Fix partial form data updates involving `ListField`" — 3 matched nodes (3M/0C/0F)
  Skipping PR #9905 — no graph overlap
  Skipping PR #9935 — no graph overlap
  Skipping PR #9936 — no graph overlap
  ✓ PR #9929: "Include `choices` param for non-editable fields" — 3 matched nodes (2M/0C

In [11]:
# ── Summary table ────────────────────────────────────────────────────
summary = []
for pr in all_prs:
    e = pr["entities"]
    summary.append({
        "repo": pr["repo_short"],
        "PR": f"#{pr['pr_number']}",
        "title": pr["pr_title"][:50],
        "files": pr["changed_files_count"],
        "modules": len(e["matched_modules"]),
        "classes": len(e["matched_classes"]),
        "functions": len(e["matched_functions"]),
        "total_nodes": pr["total_matched_nodes"],
        "unmatched": len(e["unmatched_files"]),
    })

df_summary = pd.DataFrame(summary)
print(f"\nTotal: {len(df_summary)} PRs, {df_summary['total_nodes'].sum()} matched nodes")
display(df_summary)


Total: 55 PRs, 249 matched nodes


,repo,PR,title,files,modules,classes,functions,total_nodes,unmatched
0,django,#21050,Refs #36949 -- Removed hardcoded pks in modela...,1,1,0,0,1,0
1,django,#21046,Fixed #37016 -- Avoided propagating invalid ar...,4,4,0,0,4,0
2,django,#20889,Fixed #36973 -- Made fields.E348 detect access...,3,2,0,0,2,0
3,django,#20027,Fixed #20024 -- Fixed handling of __in lookups...,3,3,0,0,3,0
4,django,#21035,Fixed #36949 -- Improved RelatedFieldWidgetWra...,4,3,0,1,4,0
5,drf,#9902,Fix partial form data updates involving `ListF...,4,3,0,0,3,0
6,drf,#9929,Include `choices` param for non-editable fields,2,2,0,1,3,0
7,drf,#9931,Prepare bug fix release 3.17.1,2,1,0,0,1,0
8,drf,#9928,Fix `HTMLFormRenderer` with empty `datetime` v...,2,2,0,0,2,0
9,drf,#9735,Preserve ordering in `MultipleChoiceField`,4,3,0,3,6,0


## 6. LLM-Only Risk Assessment

Prompt GPT-4o with **only** the PR diff and metadata — no structural/graph context.
This establishes the baseline for comparison with the RGAT-augmented version.

In [12]:
RISK_SYSTEM_PROMPT = """You are an expert software architect specialising in change risk assessment for large Python ecosystems.

You will be given a pull request diff from an open-source Django ecosystem project. Your task is to assess the **change risk** of this PR.

Risk is defined as: **Risk = Severity × Probability**

**Severity (1–10):** The intrinsic architectural importance of the components being changed.
Consider:
- Is the code a core module vs. peripheral utility?
- How many other components likely depend on this code?
- Does the change affect public APIs, base classes, or interfaces?
- Could the change break backwards compatibility?
- Is the code in a critical path (auth, data access, serialization, etc.)?

**Probability (1–10):** The likelihood that this specific change will cause issues to propagate.
Consider:
- How invasive is the change (refactor vs. bug fix vs. new feature)?
- Does it change function signatures, class hierarchies, or return types?
- Could downstream consumers of these APIs be affected?
- Are there implicit contracts being modified?
- How much of the codebase could be transitively affected?

**Respond with valid JSON only**, matching this exact schema:
```json
{
  "severity": <int 1-10>,
  "severity_reasoning": "<2-3 sentences explaining the severity score>",
  "probability": <int 1-10>,
  "probability_reasoning": "<2-3 sentences explaining the probability score>",
  "risk_score": <int: severity * probability>,
  "risk_level": "<low|medium|high|critical>",
  "key_risks": ["<risk 1>", "<risk 2>", ...],
  "affected_areas": ["<area 1>", "<area 2>", ...],
  "summary": "<1 paragraph overall risk summary>"
}
```

Risk level thresholds: low (1-15), medium (16-35), high (36-64), critical (65-100).
"""


def build_llm_only_prompt(pr_data: Dict) -> str:
    """Build the user prompt for LLM-only risk assessment."""
    entities = pr_data["entities"]
    changed_files_str = "\n".join(f"  - {f}" for f in entities["changed_files"][:20])
    if len(entities["changed_files"]) > 20:
        changed_files_str += f"\n  ... and {len(entities['changed_files']) - 20} more"

    prompt = f"""## Pull Request: {pr_data['pr_title']}
**Repository:** {pr_data['github_path']}
**PR:** #{pr_data['pr_number']} ({pr_data['pr_url']})
**Files changed:** {pr_data['changed_files_count']} (+{pr_data['additions']}/−{pr_data['deletions']})
**Merged:** {pr_data['merged_at']}

### PR Description
{pr_data['pr_body'][:1000] if pr_data['pr_body'] else '(no description)'}

### Changed Files
{changed_files_str}

### Diff
```diff
{pr_data['diff_text']}
```

Analyse this PR and provide your structured risk assessment as JSON.
"""
    return prompt


print("✓ LLM-only prompt builder ready")
print(f"System prompt length: {len(RISK_SYSTEM_PROMPT):,} chars")

✓ LLM-only prompt builder ready
System prompt length: 1,728 chars


In [13]:
def call_openai_risk_assessment(
    system_prompt: str,
    user_prompt: str,
    model: str = OPENAI_MODEL,
    max_retries: int = 3,
) -> Optional[Dict]:
    """Call OpenAI API and parse structured JSON response."""
    for attempt in range(max_retries):
        try:
            response = openai_client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=0.2,
                response_format={"type": "json_object"},
                max_tokens=1000,
            )

            content = response.choices[0].message.content
            result = json.loads(content)

            # Basic validation
            required = ["severity", "probability", "risk_score", "risk_level", "summary"]
            if all(k in result for k in required):
                result["_model"] = model
                result["_tokens"] = {
                    "prompt": response.usage.prompt_tokens,
                    "completion": response.usage.completion_tokens,
                }
                return result
            else:
                missing = [k for k in required if k not in result]
                print(f"  ⚠ Missing keys: {missing}. Retrying...")

        except json.JSONDecodeError as e:
            print(f"  ⚠ JSON parse error (attempt {attempt+1}): {e}")
        except Exception as e:
            print(f"  ⚠ API error (attempt {attempt+1}): {e}")
            time.sleep(2 ** attempt)  # exponential backoff

    return None


print("✓ OpenAI caller ready")

✓ OpenAI caller ready


In [14]:
# ── Run LLM-only risk assessment ─────────────────────────────────────
llm_only_results = []

for i, pr_data in enumerate(all_prs):
    pr_id = f"{pr_data['repo_short']}#{pr_data['pr_number']}"
    print(f"[{i+1}/{len(all_prs)}] Assessing {pr_id}: {pr_data['pr_title'][:50]}...")

    user_prompt = build_llm_only_prompt(pr_data)
    assessment = call_openai_risk_assessment(RISK_SYSTEM_PROMPT, user_prompt)

    if assessment:
        print(
            f"  → Risk: {assessment['risk_score']} ({assessment['risk_level']}) "
            f"[S={assessment['severity']} × P={assessment['probability']}]"
        )
    else:
        print(f"  ✗ Failed to get assessment")
        assessment = {"error": "API call failed"}

    llm_only_results.append({
        "pr_id": pr_id,
        "repo_short": pr_data["repo_short"],
        "pr_number": pr_data["pr_number"],
        "pr_title": pr_data["pr_title"],
        "assessment": assessment,
    })

    time.sleep(0.5)  # Rate limit courtesy

print(f"\n✓ Completed {len(llm_only_results)} LLM-only assessments")

[1/55] Assessing django#21050: Refs #36949 -- Removed hardcoded pks in modeladmin...
  → Risk: 2 (low) [S=2 × P=1]
[2/55] Assessing django#21046: Fixed #37016 -- Avoided propagating invalid argume...
  → Risk: 24 (medium) [S=6 × P=4]
[3/55] Assessing django#20889: Fixed #36973 -- Made fields.E348 detect accessor a...
  → Risk: 24 (medium) [S=6 × P=4]
[4/55] Assessing django#20027: Fixed #20024 -- Fixed handling of __in lookups wit...
  → Risk: 35 (medium) [S=7 × P=5]
[5/55] Assessing django#21035: Fixed #36949 -- Improved RelatedFieldWidgetWrapper...
  → Risk: 20 (medium) [S=5 × P=4]
[6/55] Assessing drf#9902: Fix partial form data updates involving `ListField...
  → Risk: 24 (medium) [S=6 × P=4]
[7/55] Assessing drf#9929: Include `choices` param for non-editable fields...
  → Risk: 24 (medium) [S=6 × P=4]
[8/55] Assessing drf#9931: Prepare bug fix release 3.17.1...
  → Risk: 6 (low) [S=3 × P=2]
[9/55] Assessing drf#9928: Fix `HTMLFormRenderer` with empty `datetime` value...
  → Risk: 

In [15]:
# ── Display LLM-only results ─────────────────────────────────────────
rows = []
for r in llm_only_results:
    a = r["assessment"]
    if "error" not in a:
        rows.append({
            "PR": r["pr_id"],
            "title": r["pr_title"][:40],
            "severity": a["severity"],
            "probability": a["probability"],
            "risk_score": a["risk_score"],
            "risk_level": a["risk_level"],
        })

df_llm_only = pd.DataFrame(rows)
print("LLM-Only Risk Assessments:")
display(df_llm_only.sort_values("risk_score", ascending=False))

LLM-Only Risk Assessments:


,PR,title,severity,probability,risk_score,risk_level
12,wagtail#13930,Defer validation of required fields with,7,6,42,high
9,drf#9735,Preserve ordering in `MultipleChoiceFiel,7,6,42,high
10,wagtail#14017,Store preview data in a dedicated FormSt,7,6,42,high
11,wagtail#14034,Use the same UUID for autosave audit log,7,6,42,high
29,oscar#4551,[FEAT] Add code in address models,7,6,42,high
39,simplejwt#887,fix: always stringify user_id claim,7,6,42,high
3,django#20027,Fixed #20024 -- Fixed handling of __in l,7,5,35,medium
34,filter#1698,Removed deprecated schema generation met,7,5,35,medium
46,guardian#976,Issue966,7,5,35,medium
17,netbox#21815,Fixes #21498: Fix Exception when changin,7,5,35,medium


## 7. Save Results

Save PR data and LLM-only assessments for use in Notebook 05.

In [17]:
# ── Save all experiment data ─────────────────────────────────────────
experiment_data = {
    "config": {
        "prs_per_repo": PRS_PER_REPO,
        "openai_model": OPENAI_MODEL,
        "repos": list(REPO_MAP.keys()),
    },
    "prs": all_prs,
    "llm_only_results": llm_only_results,
}

output_path = OUTPUT_DIR / "04_pr_data_and_llm_baseline.json"
with open(output_path, "w") as f:
    json.dump(experiment_data, f, indent=2, default=str)

# Persist to Drive so NB05 can pick it up across runtime resets
DRIVE_EXPERIMENT = DRIVE_DIR / "experiment_outputs"
DRIVE_EXPERIMENT.mkdir(exist_ok=True)
shutil.copy2(output_path, DRIVE_EXPERIMENT / output_path.name)

print(f"  LLM-only assessments: {len(llm_only_results)}")

print(f"✓ Saved to {output_path} (+ Drive)")
print(f"  PRs: {len(all_prs)}")
print(f"  File size: {output_path.stat().st_size / 1024:.1f} KB")

  LLM-only assessments: 55
✓ Saved to /content/rgat_project/experiment_outputs/04_pr_data_and_llm_baseline.json (+ Drive)
  PRs: 55
  File size: 521.9 KB


In [20]:
# ── Also save a compact node list for Notebook 05 ────────────────────
# This avoids Notebook 05 needing to re-parse diffs
pr_nodes = []
for pr_data in all_prs:
    e = pr_data["entities"]
    all_node_ids = (
        [nid for nid, _ in e["matched_modules"]]
        + [nid for nid, _ in e["matched_classes"]]
        + [nid for nid, _ in e["matched_functions"]]
    )
    pr_nodes.append({
        "pr_id": f"{pr_data['repo_short']}#{pr_data['pr_number']}",
        "repo_short": pr_data["repo_short"],
        "pr_number": pr_data["pr_number"],
        "node_ids": all_node_ids,
    })

nodes_path = OUTPUT_DIR / "04_pr_node_mapping.json"
with open(nodes_path, "w") as f:
    json.dump(pr_nodes, f, indent=2)

# Persist to Drive
DRIVE_EXPERIMENT = DRIVE_DIR / "experiment_outputs"
DRIVE_EXPERIMENT.mkdir(exist_ok=True)

shutil.copy2(nodes_path, DRIVE_EXPERIMENT / nodes_path.name)

total_nodes = sum(len(p["node_ids"]) for p in pr_nodes)
print(f"✓ Node mapping saved to {nodes_path} (+ Drive)")
print(f"  {len(pr_nodes)} PRs, {total_nodes} total node references")

✓ Node mapping saved to /content/rgat_project/experiment_outputs/04_pr_node_mapping.json (+ Drive)
  55 PRs, 249 total node references


---

## Summary

This notebook collected real PRs from the Django ecosystem and established the **LLM-only baseline** risk assessments. The next step (Notebook 05) will:

1. Load the trained RGAT model
2. For each PR's matched nodes, extract k-hop subgraph context and attention weights
3. Re-prompt the LLM with **diff + structural context**
4. Compare LLM-only vs. LLM+RGAT assessments side by side

The saved files:
- `experiment_outputs/04_pr_data_and_llm_baseline.json` — Full PR data + LLM-only results
- `experiment_outputs/04_pr_node_mapping.json` — Compact node ID mapping for RGAT inference